In [0]:
%run ../config/feat_squad2_config_adls

In [0]:
%run ../utils/feat_squad2_utils

In [0]:
# Importar bibliotecas
from functools import reduce

In [0]:
# Configurar as variáveis
folder_name = "vendas_raw/"
entity_name = "enderecos"
file_name_contains = f"ecommerce_{entity_name}.parquet"

In [0]:
# Listar os arquivos no container
container_files =list_files(
                        container_client=container_client
                )
print(f"Arquivos no container {container_name}:")
for file in container_files:
    print(file)

In [0]:
# Listar os arquivos ecommerce_enderecos
files = list_files(
            container_client=container_client,
            folder_name=folder_name,
            file_name_contains=file_name_contains
        )
print(f"Arquivos {file_name_contains}:")
for file in files:
    print(file)

In [0]:
# Fazer o download dos arquivos parquet parquet, convertê-los para DataFrames Spark e salvá-los em uma lista
spark_dfs = []

for file in files:
    df_file = read_parquet_to_spark_df(file_path = file)
    
    spark_dfs.append(
    df_file
    )


In [0]:
df_file = read_parquet_to_spark_df(file_path = "vendas_raw/2026/04/27/225222/ecommerce_enderecos.parquet")
display(df_file)

In [0]:
# Combinar todos os DataFrames Spark em um único DataFrame
df_clientes_agg = reduce(
    lambda x, y: x.unionByName(y),
    spark_dfs
)

In [0]:
# Mostrar o schema do Dataframe
df_clientes_agg.printSchema()

In [0]:
total_records = df_clientes_agg.count()

print(f"Total de registros: {total_records:,}")

In [0]:
# Analizar a quantidade de valores nulos em cada tabela e a porcentagem de nulos em relação ao total	
from pyspark.sql.functions import col

nulos = []

for c in df_clientes_agg.columns:
    qtd_null = df_clientes_agg.filter(col(c).isNull()).count()
    qtd_none = df_clientes_agg.filter(col(c) == "None").count()
    qtd = qtd_null + qtd_none
    
    perc = round((qtd / total_records) * 100, 2)

    nulos.append((c, qtd, perc))

nulos_df = spark.createDataFrame(
    nulos,
    ["coluna", "qtd_nulos", "percentual"]
)

display(
    nulos_df.orderBy(col("percentual").desc())
)